In [1]:
# [Cell 0] Python 인터프리터 사전 확인
# 노트북이 올바른 가상환경(MLOps2venv)의 Python을 사용하는지 확인합니다.
# 경로에 "MLOps2venv" 가 포함되어 있어야 합니다.
import sys
print(sys.executable)   # Python 실행 파일 경로
print(sys.version)      # Python 버전

/Users/macminim4/Aiffel02/MLOps02/MLOps2venv/bin/python
3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.6.3.2)]


In [2]:
# [Cell 1] 작업 디렉토리 및 Python 경로 설정
# ─────────────────────────────────────────────────────────────────────────
# 왜 필요한가?
#   1. %%writefile 명령은 현재 작업 디렉토리(CWD) 기준으로 파일을 생성합니다.
#      → lecture07/ 를 CWD로 설정해야 app/chatbot_model.py 가 올바른 위치에 생성됩니다.
#   2. "from app.auth import ..." 같은 import는
#      lecture07/ 가 sys.path에 있어야 합니다.
# ─────────────────────────────────────────────────────────────────────────
import os, sys

NOTEBOOK_DIR = "/Users/macminim4/Aiffel02/MLOps02/lecture07"
os.chdir(NOTEBOOK_DIR)                    # 1. CWD를 노트북 폴더로 변경

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)      # 2. app.* import 가능하게 경로 추가

print(f" 작업 디렉토리: {os.getcwd()}")
print(f" Python 경로 추가: {sys.path[0]}")
print()
print("현재 폴더 구조:")
for item in sorted(os.listdir(NOTEBOOK_DIR)):
    if item.startswith("."):      # .DS_Store 등 숨김 파일 제외
        continue
    full = os.path.join(NOTEBOOK_DIR, item)
    prefix = "" if os.path.isdir(full) else ""
    print(f"  {prefix} {item}")

 작업 디렉토리: /Users/macminim4/Aiffel02/MLOps02/lecture07
 Python 경로 추가: /Users/macminim4/Aiffel02/MLOps02/lecture07

현재 폴더 구조:
   01_project_flow.md
   02_system_structure.md
   03_cell_by_cell.md
   app
   frontend
   모델배포개론07.ipynb
   한국어GPT챗봇배포_submit.ipynb


In [3]:
# [Cell 2] PyTorch 설치 확인 및 GPU/MPS 환경 체크
# transformers가 설치되어 있지 않다면
# !pip install transformers accelerate -q

import torch  # PyTorch: 딥러닝 핵심 라이브러리 (텐서 연산 + 자동 미분)
print(f"PyTorch 버전: {torch.__version__}")
print()
print(f"CUDA  (NVIDIA GPU) : {torch.cuda.is_available()}")
print(f"MPS   (Apple  GPU) : {torch.backends.mps.is_available()}")  # M1/M2/M3/M4 Mac
print()
# CUDA → MPS → CPU 우선순위로 사용 가능한 최적 디바이스 선택
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"▶ 사용할 device: {device}")

PyTorch 버전: 2.4.1

CUDA  (NVIDIA GPU) : False
MPS   (Apple  GPU) : True

▶ 사용할 device: mps


In [4]:
# [Cell 3] GPU/CPU에 따라 KoGPT 모델 선택
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

# GPU 환경이면 큰 모델, CPU 환경이면 작은 모델
# CUDA(NVIDIA GPU) 또는 MPS(Apple Silicon) → 1.2B 파라미터 대형 모델
# CPU only → 125M 파라미터 소형 모델 (M4 Mac은 MPS 경로로 대형 모델 사용)
gpu_available = torch.cuda.is_available() or torch.backends.mps.is_available()
MODEL_NAME = "skt/ko-gpt-trinity-1.2B-v0.5" if gpu_available else "skt/kogpt2-base-v2"
device_type = "CUDA" if torch.cuda.is_available() else ("MPS" if torch.backends.mps.is_available() else "CPU")
print(f"모델: {MODEL_NAME}")
print(f"디바이스: {device_type}")
print("모델 다운로드 중... (최초 실행 시 시간이 걸립니다)")

/Users/macminim4/Aiffel02/MLOps02/MLOps2venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델: skt/ko-gpt-trinity-1.2B-v0.5
디바이스: MPS
모델 다운로드 중... (최초 실행 시 시간이 걸립니다)


In [5]:
# [Cell 4] 토크나이저와 모델 로드 후 평가 모드 전환
# CUDA → MPS → CPU 우선순위로 디바이스 선택
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"사용 디바이스: {device}")

# PreTrainedTokenizerFast: 텍스트 → 정수 토큰 ID 변환
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME)    # *your code* — 토크나이저 로드
# use_safetensors=True: pickle 보안 취약점을 피하는 안전한 가중치 포맷
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, use_safetensors=True)  # *your code* — 모델 로드
model = model.to(device)  # 모델 가중치를 선택된 디바이스 메모리로 이동
model.eval()              # 드롭아웃 OFF → 추론 모드 (결과 재현성 보장)

print(f" 모델 로드 완료")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

사용 디바이스: mps


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 39273.26it/s]
GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 모델 로드 완료
파라미터 수: 1,162,556,160


In [6]:
# [Cell 5] 단일 프롬프트 텍스트 생성 테스트
# 단일 프롬프트로 텍스트 생성
prompt = "마르크스는 위대한가"

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():  # 기울기 계산 비활성화 → 메모리 절약, 속도 향상
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,           # *your code* — 최대 생성 토큰 수           # 이 값을 늘리면 더 긴 문장 생성
        temperature=0.8,            # 낮을수록 보수적(반복적), 높을수록 창의적(무작위)
        top_k=50,                    # 상위 50개 토큰 후보 중에서만 샘플링
        top_p=0.9,                   # 누적 확률 90% 이내 토큰만 허용 (nucleus sampling)
        do_sample=True,             # True: 확률적 샘플링 / False: 항상 가장 확률 높은 토큰
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"프롬프트: {prompt}")
print(f"생성 결과: {generated_text}")

프롬프트: 마르크스는 위대한가
생성 결과: 마르크스는 위대한가 아닌가? 그렇지 않다면 그 어떤 위대한 철학자도 우리 시대와 우리 시대의 철학을 제대로 이해하기 어렵다. 이 책은 이러한 문제를 해결하고 철학을 보다 더 풍요롭게 하기 위해 철학자들이 쓴 책이다. 철학자들이 어떻게 세상을 보았고, 무엇을 말했으며, 세상을 어떻게


In [7]:
# [Cell 6] 멀티턴 대화 프롬프트 구성 및 생성 테스트
# 대화 기록을 하나의 프롬프트로 결합
conversation = [
    "사용자: 안녕하세요!",
    "봇: 안녕하세요! 무엇을 도와드릴까요?",
    "사용자: 공부와 운동 어떤게 더 좋아?",
]

prompt = "\n".join(conversation) + "\n봇:"        # *your code* — 대화 기록 결합

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(
        input_ids, max_new_tokens=50, temperature=0.8,
        top_k=50, do_sample=True, pad_token_id=tokenizer.eos_token_id,
    )

full_response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
bot_response = full_response[len(prompt):].split("사용자:")[0].strip()

print(f"봇 응답: {bot_response}")

봇 응답: 공부와 운동 둘다 좋습니다!  
 : 위키백과:학생 대모험을 통해 학생 대모험의 모든것에 대해 알 수 있습니다. 그리고 위키백과:학생 대모험 참고!
 : 위키백과:학생 대모험의 모든


In [8]:
%%writefile app/chatbot_model.py
# [Cell 7] ChatbotModel 클래스 → app/chatbot_model.py 저장
"""
Day 7 - 한국어 GPT 챗봇 모델
"""
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel


class ChatbotModel:
    """Hugging Face 한국어 GPT 모델을 로드하고 텍스트를 생성합니다."""

    def __init__(self, model_name: str = "skt/kogpt2-base-v2"):
        self.device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")  # MPS: Apple Silicon GPU

        self.tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
        self.model = GPT2LMHeadModel.from_pretrained(model_name, use_safetensors=True)
        self.model = self.model.to(self.device)
        self.model.eval()

        self.model_name = model_name

    def generate_response(
        self,
        messages: list[dict],
        max_new_tokens: int = 100,
        temperature: float = 0.8,
        top_k: int = 50,
        top_p: float = 0.9,
        repetition_penalty: float = 1.3,   # 반복 억제: 1.0=없음, 높을수록 반복 감소
        no_repeat_ngram_size: int = 3,      # 이 크기의 n-gram이 두 번 나오면 차단
    ) -> str:
        """
        대화 기록을 받아 응답을 생성합니다.

        Args:
            messages: [{"role": "user", "content": "안녕"}, {"role": "bot", "content": "안녕하세요!"}, ...]
            max_new_tokens: 최대 생성 토큰 수
            temperature: 생성 다양성
        Returns:
            생성된 응답 텍스트
        """
        # 대화 기록 → 프롬프트 구성
        prompt = self._build_prompt(messages)              # *your code* — 프롬프트 구성

        # 토크나이징
        input_ids = self.tokenizer.encode(
            prompt, return_tensors="pt"
        ).to(self.device)

        # 토큰 수 제한 (모델 최대 길이 초과 방지)
        max_length = getattr(self.model.config, "n_positions", 1024)
        if input_ids.shape[1] > max_length - max_new_tokens:
            # 최근 대화만 유지
            input_ids = input_ids[:, -(max_length - max_new_tokens):]

        # 텍스트 생성
        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,             # *your code* — 생성 파라미터
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                do_sample=True,
                repetition_penalty=repetition_penalty,     # 반복 단어/구절 억제
                no_repeat_ngram_size=no_repeat_ngram_size, # 동일 n-gram 재출현 차단
                pad_token_id=self.tokenizer.eos_token_id,
            )

        # 디코딩: 생성된 부분만 추출
        full_text = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        response = full_text[len(prompt):].strip()

        # "사용자:" 이후 텍스트가 나오면 거기까지만 자름
        if "사용자:" in response:
            response = response.split("사용자:")[0].strip()

        return response if response else "(응답을 생성하지 못했습니다)"

    def _build_prompt(self, messages: list[dict]) -> str:
        """대화 기록을 프롬프트 문자열로 변환합니다."""
        lines = []
        for msg in messages:
            role = "사용자" if msg["role"] == "user" else "봇"
            lines.append(f"{role}: {msg['content']}")
        lines.append("봇:")  # 모델이 이어서 생성하도록
        return "\n".join(lines)

Overwriting app/chatbot_model.py


In [9]:
%%writefile app/chatbot_schemas.py
# [Cell 8] Pydantic 요청/응답 스키마 → app/chatbot_schemas.py 저장
"""
Day 7 - 챗봇 API 스키마
"""
from pydantic import BaseModel, Field
from typing import Optional


class Message(BaseModel):
    """단일 대화 메시지"""
    role: str = Field(..., description="역할: 'user' 또는 'bot'")
    content: str = Field(..., min_length=1, description="메시지 내용")


class ChatRequest(BaseModel):
    """챗봇 요청"""
    messages: list[Message] = Field(
        ...,
        min_length=1,
        description="대화 기록. 마지막 메시지가 사용자의 현재 입력.",
    )
    max_new_tokens: int = Field(default=100, ge=10, le=500)     # *your code* — 범위 제한
    temperature: float = Field(default=0.8, gt=0.0, le=2.0)

    model_config = {
        "json_schema_extra": {
            "examples": [
                {
                    "messages": [
                        {"role": "user", "content": "안녕하세요!"}
                    ],
                    "max_new_tokens": 100,
                    "temperature": 0.8,
                }
            ]
        }
    }


class ChatResponse(BaseModel):
    """챗봇 응답"""
    success: bool = Field(description="성공 여부")
    response: str = Field(description="생성된 응답 텍스트")
    model_name: str = Field(description="사용된 모델")
    user: Optional[str] = Field(default=None, description="인증된 사용자")

Overwriting app/chatbot_schemas.py


In [10]:
%%writefile app/chatbot_api.py
# [Cell 9] FastAPI 챗봇 서버 → app/chatbot_api.py 저장
"""
Day 7 - 한국어 챗봇 FastAPI 서버
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import FastAPI, Depends, HTTPException

from app.chatbot_schemas import ChatRequest, ChatResponse
from app.chatbot_model import ChatbotModel
from app.auth import verify_api_key
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


# ===== 설정 =====
logger = setup_logger("chatbot_api")

app = FastAPI(
    title="Korean Chatbot API",
    description="한국어 GPT 기반 멀티턴 챗봇 API (인증 필요)",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="chatbot")

# ===== 모델 로드 =====
chatbot = None

@app.on_event("startup")
async def startup():
    global chatbot
    import torch
    model_name = "skt/ko-gpt-trinity-1.2B-v0.5" if (torch.cuda.is_available() or torch.backends.mps.is_available()) else "skt/kogpt2-base-v2"
    logger.info(f"챗봇 모델 로드 중: {model_name}")
    chatbot = ChatbotModel(model_name)                # *your code* — ChatbotModel 인스턴스 생성
    logger.info("모델 로드 완료")


def run_chat(messages, max_new_tokens, temperature):
    """별도 스레드에서 실행되는 추론 함수"""
    if chatbot is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return chatbot.generate_response(
        messages=[m.model_dump() for m in messages],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
    )


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if chatbot else "loading",
        "model": chatbot.model_name if chatbot else None,
    }


@app.post("/chat", response_model=ChatResponse, tags=["Chat"])
async def chat(
    request: ChatRequest,
    user: str = Depends(verify_api_key),              # *your code* — 인증 적용
):
    """대화 기록을 받아 응답을 생성합니다."""
    if chatbot is None:
        raise HTTPException(status_code=503, detail="모델이 아직 로드되지 않았습니다.")

    logger.info(f"채팅 요청 — 사용자: {user}, 메시지 수: {len(request.messages)}")

    try:
        loop = asyncio.get_event_loop()
        response_text = await loop.run_in_executor(    # *your code* — 비동기 추론
            inference_executor,
            run_chat,
            request.messages,
            request.max_new_tokens,
            request.temperature,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"텍스트 생성 실패: {str(e)}")

    return ChatResponse(
        success=True,
        response=response_text,
        model_name=chatbot.model_name,
        user=user,
    )

Overwriting app/chatbot_api.py


In [11]:
# [Cell 10] 서버 실행 직전 모델 참조 재확인
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_NAME)    # *your code* — 토크나이저 로드
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, use_safetensors=True)  # *your code* — 모델 로드
model = model.to(device)
model.eval()

print(f" 모델 로드 완료")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 10215.84it/s]
GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 모델 로드 완료
파라미터 수: 1,162,556,160


In [12]:
# [Cell 11] FastAPI 챗봇 서버 백그라운드 시작
#  이전 서버가 실행 중이면 커널을 재시작하세요.
#  모델 다운로드에 시간이 걸릴 수 있습니다 (최초 1회).

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()  # Jupyter는 이미 이벤트 루프 실행 중 → 중첩 허용

def run_server():
    import asyncio
    asyncio.set_event_loop(asyncio.new_event_loop())  # uvloop 스레드 이슈 해결
    uvicorn.run("app.chatbot_api:app", host="0.0.0.0", port=8000, loop="asyncio")

server_thread = threading.Thread(target=run_server, daemon=True)  # 메인 스레드 종료 시 서버도 자동 종료
server_thread.start()
time.sleep(10)   # 모델 로드 대기 (첫 실행 시 더 오래 걸릴 수 있음)   # 모델 첫 로드는 수십 초 걸릴 수 있음 (필요 시 늘려도 됨)
print(" 서버 시작됨: http://localhost:8000")

INFO:     Started server process [70584]
INFO:     Waiting for application startup.


2026-04-06 17:47:55 INFO     [chatbot_api] 챗봇 모델 로드 중: skt/ko-gpt-trinity-1.2B-v0.5


Loading weights: 100%|██████████| 292/292 [00:00<00:00, 3766.82it/s]
GPT2LMHeadModel LOAD REPORT from: skt/ko-gpt-trinity-1.2B-v0.5
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 서버 시작됨: http://localhost:8000
2026-04-06 17:48:05 INFO     [chatbot_api] 모델 로드 완료


INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [13]:
# [Cell 12] 헬스체크 — 서버 및 모델 상태 확인
import requests, json

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"헬스체크: {resp.json()}")

헬스체크: {'status': 'healthy', 'model': 'skt/ko-gpt-trinity-1.2B-v0.5'}


In [14]:
# [Cell 13] 싱글턴 대화 API 테스트
# 싱글턴 대화 테스트
resp = requests.post(
    "http://localhost:8000/chat",
    json={
        "messages": [
            {"role": "user", "content": "안녕하세요!"}
        ],
    },
    headers={"X-API-Key": "test-key-001"},
)
result = resp.json()
print(f"상태: {resp.status_code}")
print(f"응답: {result['response']}")

상태: 200
응답: Jisu (천리주단기)
 ::# <span style="font-size:110%;color:red;">Park4223</span>(토론) 2015년 11월 24일 (일) 23:01 (KST)
 ::::::::  다른 언어판에는 없는 영어 위키백과에 있는 계정 이름만 가져왔습니다. --양념파닭 (토론) 2015년 12월 1일 (수) 13:52 (KST)


In [15]:
# [Cell 14] 멀티턴 대화 API 테스트
# 멀티턴 대화 테스트
resp = requests.post(
    "http://localhost:8000/chat",
    json={
        "messages": [
            {"role": "user", "content": "안녕하세요!"},
            {"role": "bot", "content": result["response"]},    # 이전 응답 포함
            {"role": "user", "content": "오늘 뭐 하면 좋을까?"},
        ],
    },
    headers={"X-API-Key": "test-key-001"},
)
result2 = resp.json()
print(f"멀티턴 응답: {result2['response']}")

멀티턴 응답: 오늘뭐하면좋을까
 # <br />이전에 차단된 사용자입니다.--175.197.33.164 (토론|메일)
 :해당 사용자의 토론란에서 한 번 더 확인해주세요. 그리고 해당 사용자가 다중계정인지 아닌지 꼭 확인해 주시고요. 또한 '저는 편집하지 않습니다' 라는 것은 자신이 편집할 수 없다는 의미라기보다는, 현재 자신의 기여를 중단한다는 것을 의미하는 것 같습니다. 그런 의미에서 제가 그 내용을 수정하는 것으로 토론을


In [16]:
%%writefile frontend/app_chatbot.py
# [Cell 15] Streamlit 챗봇 UI → frontend/app_chatbot.py 저장
"""
Day 7 - 한국어 GPT 챗봇 대시보드
"""
import streamlit as st
import requests


# ===== 페이지 설정 =====
st.set_page_config(
    page_title="한국어 챗봇",
    page_icon="",
    layout="centered",
)

API_BASE = "http://localhost:8000"


# ===== API 호출 =====
def call_chat_api(messages, api_key, max_new_tokens=100, temperature=0.8):
    """챗봇 API를 호출합니다."""
    try:
        resp = requests.post(
            f"{API_BASE}/chat",
            json={
                "messages": messages,
                "max_new_tokens": max_new_tokens,
                "temperature": temperature,
            },
            headers={"X-API-Key": api_key},            # *your code* — 인증 헤더
            timeout=60,   # 텍스트 생성은 시간이 걸릴 수 있음
        )
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.ConnectionError:
        st.error(" **서버에 연결할 수 없습니다.**")
        return None
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 401:
            st.error(" **인증 실패.** API Key를 확인하세요.")
        else:
            st.error(f" **서버 에러** (HTTP {e.response.status_code})")
        return None
    except Exception as e:
        st.error(f" **오류:** {type(e).__name__}")
        return None


# ===== 사이드바 =====
with st.sidebar:
    st.header(" 설정")

    api_key = st.text_input("API Key", value="test-key-001", type="password")

    st.divider()

    max_tokens = st.slider("최대 생성 토큰", 10, 300, 100, step=10)
    temperature = st.slider("Temperature", 0.1, 2.0, 0.8, step=0.1)

    st.divider()

    # 서버 상태
    try:
        health = requests.get(f"{API_BASE}/health", timeout=3).json()
        if health.get("status") == "healthy":
            st.success(f" 서버 연결됨")
            st.caption(f"모델: {health.get('model', 'N/A')}")
        else:
            st.warning(" 모델 로딩 중...")
    except Exception:
        st.error(" 서버 연결 실패")

    st.divider()

    if st.button("대화 초기화"):
        st.session_state["chat_messages"] = []         # *your code* — 대화 기록 초기화
        st.rerun()

    st.caption("Korean GPT Chatbot v1.0")


# ===== 대화 기록 초기화 =====
if "chat_messages" not in st.session_state:
    st.session_state["chat_messages"] = []


# ===== 메인 영역 =====
st.title(" 한국어 챗봇")
st.write("한국어 GPT 모델과 대화해 보세요.")

# 기존 대화 기록 표시
for msg in st.session_state["chat_messages"]:
    with st.chat_message(msg["role"]):                 # *your code* — 채팅 메시지 표시
        st.write(msg["content"])

# 사용자 입력
user_input = st.chat_input("메시지를 입력하세요...")     # *your code* — 채팅 입력란

if user_input:
    # 사용자 메시지 추가 및 표시
    st.session_state["chat_messages"].append({
        "role": "user",
        "content": user_input,
    })
    with st.chat_message("user"):
        st.write(user_input)

    # API 호출
    with st.chat_message("assistant"):
        with st.spinner("생성 중..."):
            # 대화 기록을 API 형식으로 변환
            api_messages = []
            for msg in st.session_state["chat_messages"]:
                role = "user" if msg["role"] == "user" else "bot"
                api_messages.append({"role": role, "content": msg["content"]})

            result = call_chat_api(
                messages=api_messages,
                api_key=api_key,
                max_new_tokens=max_tokens,
                temperature=temperature,
            )

        if result and result.get("success"):
            bot_response = result["response"]
            st.write(bot_response)

            # 봇 응답을 대화 기록에 추가
            st.session_state["chat_messages"].append({
                "role": "assistant",
                "content": bot_response,
            })
        else:
            st.write("응답을 생성하지 못했습니다.")

Overwriting frontend/app_chatbot.py


In [17]:
# [Cell 16] 통합 테스트용 서버 재시작
#  이전 서버가 실행 중이면 커널을 재시작하세요.
#  모델 다운로드에 시간이 걸릴 수 있습니다.

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    import asyncio
    asyncio.set_event_loop(asyncio.new_event_loop())  # uvloop 스레드 이슈 해결
    uvicorn.run("app.chatbot_api:app", host="0.0.0.0", port=8000, loop="asyncio")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(10)
print(" 백엔드: http://localhost:8000")

INFO:     Started server process [70584]
INFO:     Waiting for application startup.


2026-04-06 17:50:56 INFO     [chatbot_api] 챗봇 모델 로드 중: skt/ko-gpt-trinity-1.2B-v0.5


Task exception was never retrieved
future: <Task finished name='Task-1' coro=<Server.serve() done, defined at /Users/macminim4/Aiffel02/MLOps02/MLOps2venv/lib/python3.11/site-packages/uvicorn/server.py:67> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/Users/macminim4/Aiffel02/MLOps02/MLOps2venv/lib/python3.11/site-packages/uvicorn/server.py", line 162, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_2/Frameworks/Python.framework/Versions/3.11/lib/python3.11/asyncio/base_events.py", line 1536, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_2/Frameworks/Python.framework/Versions/3.11/lib/python3.11/threading.p

 백엔드: http://localhost:8000


In [18]:
# [Cell 17] Streamlit UI 프로세스 백그라운드 시작
import subprocess
proc = subprocess.Popen(
    ["streamlit", "run", "frontend/app_chatbot.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(" 프론트엔드: http://localhost:8501")

 프론트엔드: http://localhost:8501
2026-04-06 17:52:30 INFO     [chatbot_api] 모델 로드 완료


INFO:     Application startup complete.
ERROR:    [Errno 48] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [19]:
# [Cell 18] 통합 테스트 — API 및 상수 설정
import requests, json

API_BASE = "http://localhost:8000"
VALID_KEY = "test-key-001"

print("=" * 60)
print("  통합 테스트")
print("=" * 60)

  통합 테스트


In [20]:
# [Cell 19] 테스트 1: API Key 인증 검증 (주석 해제 후 실행)
print("\n[테스트 1] 인증")

# 인증 없이
resp = requests.post(f"{API_BASE}/chat",
    json={"messages": [{"role": "user", "content": "안녕"}]})
print(f"  인증 없음    → HTTP {resp.status_code}")

# 잘못된 키
resp = requests.post(f"{API_BASE}/chat",
    json={"messages": [{"role": "user", "content": "안녕"}]},
    headers={"X-API-Key": "wrong-key"})
print(f"  잘못된 키    → HTTP {resp.status_code}")

# 올바른 키
resp = requests.post(f"{API_BASE}/chat",
    json={"messages": [{"role": "user", "content": "안녕"}]},
    headers={"X-API-Key": VALID_KEY})
print(f"  올바른 키    → HTTP {resp.status_code}")
if resp.status_code == 200:
    print(f"  응답: {resp.json()['response'][:50]}...")


[테스트 1] 인증
  인증 없음    → HTTP 401
  잘못된 키    → HTTP 401
  올바른 키    → HTTP 200
  응답: 봇/비자유 저작물 업로드 요청에 따른 삭제 신청 틀이 부착되어 있는 경우, <span st...


In [21]:
# [Cell 20] 테스트 2: 멀티턴 대화 흐름 (주석 해제 후 실행)
print("\n[테스트 2] 멀티턴 대화")

messages = []
turns = ["안녕하세요!", "오늘 뭐 하면 좋을까?", "맛있는 거 추천해줘"]

for user_msg in turns:
    messages.append({"role": "user", "content": user_msg})

    resp = requests.post(f"{API_BASE}/chat",
        json={"messages": messages, "max_new_tokens": 50},
        headers={"X-API-Key": VALID_KEY})

    result = resp.json()
    bot_msg = result["response"]
    messages.append({"role": "bot", "content": bot_msg})    # *your code* — 봇 응답 추가

    print(f"  사용자: {user_msg}")
    print(f"  봇:    {bot_msg[:60]}...")
    print()

print(f"  총 대화 턴: {len(messages) // 2}")


[테스트 2] 멀티턴 대화
  사용자: 안녕하세요!
  봇:    봇/등록 요청
 위키백과에 도움이 될만한 편집이 필요한 경우라면 언제든지 기여하실 수 있습니다! --Min'...

  사용자: 오늘 뭐 하면 좋을까?
  봇:    오늘뭐하면좋을까? --121.149.22.85 (토론) 2012년 10월 27일 (금) 14:04 (KST)...

  사용자: 맛있는 거 추천해줘
  봇:    맛있게 먹는 방법 좀 알려주세요
 사용자토론: 맛있어 보이는 메뉴 좀 달아주세요.
 사용자토론 문서 훼손 및...

  총 대화 턴: 3


In [22]:
# [Cell 21] 테스트 3: 입력값 범위 검증 (주석 해제 후 실행)
print("[테스트 3] 입력 검증")

# 빈 메시지 목록
resp = requests.post(f"{API_BASE}/chat",
    json={"messages": []},
    headers={"X-API-Key": VALID_KEY})
print(f"  빈 메시지    → HTTP {resp.status_code}")

# temperature 범위 초과
resp = requests.post(f"{API_BASE}/chat",
    json={"messages": [{"role": "user", "content": "테스트"}], "temperature": 5.0},
    headers={"X-API-Key": VALID_KEY})
print(f"  temperature 초과 → HTTP {resp.status_code}")

[테스트 3] 입력 검증
  빈 메시지    → HTTP 422
  temperature 초과 → HTTP 422


In [23]:
# [Cell 22] 테스트 4: 동시 요청 처리 성능 (주석 해제 후 실행)
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_chat(i):
    start = time.time()
    resp = requests.post(f"{API_BASE}/chat",
        json={"messages": [{"role": "user", "content": f"질문 {i}번입니다"}], "max_new_tokens": 30},
        headers={"X-API-Key": VALID_KEY}, timeout=60)
    return {"id": i+1, "elapsed": round(time.time()-start, 1), "status": resp.status_code}

print("\n[테스트 4] 동시 요청 (4개)")
start = time.time()
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(send_chat, i) for i in range(4)]
    results = [f.result() for f in as_completed(futures)]

total = round(time.time() - start, 1)
for r in sorted(results, key=lambda x: x["id"]):
    print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
print(f"  전체: {total}초")


[테스트 4] 동시 요청 (4개)
  요청 #1: 7.1초 (HTTP 200)
  요청 #2: 5.3초 (HTTP 200)
  요청 #3: 7.2초 (HTTP 200)
  요청 #4: 5.2초 (HTTP 200)
  전체: 7.2초
